In [ ]:
# Install LangChain and Chroma (run this cell first if you get ModuleNotFoundError for langchain_chroma)
import sys
import subprocess
packages = [
    "langchain",
    "langchain-chroma",
    "langchain-openai",
    "chromadb",
    "scikit-learn",
    "plotly",
    "PyPDF2",
    "python-docx",
]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)
print("LangChain and dependencies installed.")

In [121]:
import os
import glob
import base64
import io
from datetime import datetime, timedelta
from pathlib import Path
from email.utils import parsedate_to_datetime

# Third-party imports
from dotenv import load_dotenv
import gradio as gr

# Google API imports
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# Visualization imports
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Document processing imports
import PyPDF2
from docx import Document as DocxDocument


ModuleNotFoundError: No module named 'langchain_chroma'

In [3]:
# Install required Google API packages
import sys
import subprocess

packages = ['google-api-python-client', 'google-auth-oauthlib', 'google-auth-httplib2']
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + packages)
    print("Packages installed successfully!")
except subprocess.CalledProcessError:
    # If SSL fails, try with trusted-host flag
    print("Standard install failed, trying with --trusted-host...")
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', 
        '--trusted-host', 'pypi.org',
        '--trusted-host', 'files.pythonhosted.org'
    ] + packages)
    print("Packages installed successfully!")

  Using cached google_api_python_client-2.190.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth_httplib2-0.3.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached httplib2-0.31.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached google_auth-2.48.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached google_api_core-2.29.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached googleapis_common_protos-1.72.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached proto_plus-1.27.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pyasn1-0.6.2-py3-none-any.whl.metadata (8.4 kB)
Using cached google_api_python_client-2.190.0-py3-none-any.whl (14.7 MB)
Using cached google_auth_httplib2-0.3.0-py3-none-any.whl (9.5 kB)
Using cached go


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [112]:
# The imports
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import os
import gradio as gr
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel
from pydantic import BaseModel, Field
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from time import time
from datetime import datetime, timezone, timedelta, date
from zoneinfo import ZoneInfo


In [113]:
# The usual starting point
load_dotenv(override=True)
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [114]:
SCOPES = ['https://www.googleapis.com/auth/calendar.events']
CREDENTIALS_FILE = 'credentials.json'
TOKEN_FILE = 'token.json'

def get_calendar_service():
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_FILE, 'w') as token:
            token.write(creds.to_json())
    return build('calendar', 'v3', credentials=creds)

class calendarEvent(BaseModel):     
    name: str = Field(description="The name of the event")

    onDate: datetime = Field(description="The date that the event takes place. The datetime structure should be like this: 2026-02-18T15:00:00+01:00")

    description: str = Field(description="What is this event about. Description of it")

    duration: int = Field(description="For how long this event takes place")   

class day(BaseModel):
    day: datetime = Field(description="The calendar day that the user is requesting to see all the events that he has on his calendar")

def generate_google_calendar_event(event: calendarEvent) -> Dict:
    return {
            "summary": event.name,
            "description": event.description,
            "start": {
                "dateTime": event.onDate.isoformat(),
                "timeZone": "Europe/Berlin",
            },
            "end": {
                "dateTime": (event.onDate + timedelta(minutes=event.duration)).isoformat(),
                "timeZone": "Europe/Berlin",
            },
     }

In [115]:
service =get_calendar_service()

def google_calendar_event() -> Dict:
    return {
    "summary": "NAme",
    "description": "event.description",
    "start": {
        "dateTime": "2026-02-18T15:00:00+01:00",
        "timeZone": "Europe/Berlin"
    },
    "end": {
        "dateTime": "2026-02-18T16:00:00+01:00",
        "timeZone": "Europe/Berlin"
    }
}

event = google_calendar_event()

In [116]:
class calendarEvent(BaseModel):     
    name: str = Field(description="The name of the event")

    onDate: datetime = Field(description="The date that the event takes place. The datetime structure should be like this: 2026-02-18T15:00:00+01:00")

    description: str = Field(description="What is this event about. Description of it")

    duration: int = Field(description="For how long this event takes place")   

class day(BaseModel):
    day: datetime = Field(description="The calendar day that the user is requesting to see all the events that he has on his calendar. The datetime structure should be like this: 2026-02-18T15:00:00+01:00")


In [ ]:
@function_tool
def event_notifier(currentDay: day):
    """Get events for a specific day from Google Calendar."""
    from datetime import time as dt_time
        
    berlin_tz = ZoneInfo("Europe/Berlin")
    requested_date = currentDay.day.date() if isinstance(currentDay.day, datetime) else currentDay.day
    
    # Start and end of day in Berlin timezone (aware datetimes)
    start_of_day = datetime.combine(requested_date, dt_time.min, tzinfo=berlin_tz)
    end_of_day = datetime.combine(requested_date, dt_time.max, tzinfo=berlin_tz)
    
    print(f"Searching events from {start_of_day.isoformat()} to {end_of_day.isoformat()}")
    
    service = get_calendar_service()
    
    events_result = service.events().list(
        calendarId='mbehrends1804@gmail.com',
        timeMin=start_of_day.isoformat(),
        timeMax=end_of_day.isoformat(),
        singleEvents=True,
        orderBy='startTime'
    ).execute()
    
    events = events_result.get('items', [])
    
    if not events:
        return f"No events found for {requested_date.strftime('%Y-%m-%d')}."
    
    event_list = []
    for event in events:
        start = event['start'].get('dateTime', event['start'].get('date'))
        summary = event.get('summary', 'No title')
        event_list.append(f"- {summary} at {start}")
    
    return f"Events for {requested_date.strftime('%Y-%m-%d')}:\n" + "\n".join(event_list)
    
@function_tool
def event_creator(event: calendarEvent):
    body = generate_google_calendar_event(event)
    service = get_calendar_service()
    created_event = service.events().insert(calendarId="mbehrends1804@gmail.com", body=body).execute()
    
    return f"Event '{created_event['summary']}' scheduled on {created_event['start']['dateTime']}."

tools = [event_creator, event_notifier]

In [120]:
instructions = """You are the assistant of Paris Zachoudis and Meike Behrends, your responsibility is to help them with their work 
and to be a friendly assistant.
Your first task is to add for them events in their calendar when they ask you to. Their shared calendar is a google calendar.
If the user wants to create an event,you MUST call the event_creator tool.

Extract:
- name
- date
- description
- duration

If any required information is missing,
ask the user before calling the tool.

Do NOT respond with text if the user wants to create an event.
Always call the tool.

Your second task is to tell the user the events he has for the day he requested to know about. If the user requests to know what the events or what he has to do for tomorrow
call the event_notifier_tool
Extract:
- date
"""

assistant_agent = Agent(
    name="Calendar Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini"
    )


async def chat(message, history):
    now_str = datetime.now(ZoneInfo("Europe/Berlin")).strftime("%Y-%m-%dT%H:%M:%S%z")
    full_message = (
        f"Current datetime: {now_str}\n"
        f"User message: {message}"
    )

    result = await Runner.run(assistant_agent, full_message)

    return result.final_output

demo = gr.ChatInterface(
    fn=chat,
    title="Multi-Agent Chat with OpenAI Agent SDK"
)

gr.ChatInterface(chat, type="messages").launch()

/Users/paris_z/projects/agents/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7893
* To create a public link, set `share=True` in `launch()`.


User requested to learn the events of the day: 2026-02-19 00:00:00+01:00
Searching events from 2026-02-19T00:00:00+01:00 to 2026-02-19T23:59:59.999999+01:00
User requested to learn the events of the day: 2026-02-20 20:58:30+01:00
Searching events from 2026-02-20T00:00:00+01:00 to 2026-02-20T23:59:59.999999+01:00
User requested to learn the events of the day: 2026-02-19 00:00:00+01:00
Searching events from 2026-02-19T00:00:00+01:00 to 2026-02-19T23:59:59.999999+01:00
User requested to learn the events of the day: 2026-02-20 20:58:56+01:00
Searching events from 2026-02-20T00:00:00+01:00 to 2026-02-20T23:59:59.999999+01:00
User requested to learn the events of the day: 2026-02-19 00:00:00+01:00
Searching events from 2026-02-19T00:00:00+01:00 to 2026-02-19T23:59:59.999999+01:00
User requested to learn the events of the day: 2026-02-20 00:00:00+01:00
Searching events from 2026-02-20T00:00:00+01:00 to 2026-02-20T23:59:59.999999+01:00
User requested to learn the events of the day: 2026-02-21 

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


2026-02-18T20:06:17+0100


## Now go and look at the **trace**

But actually **don't**. `OpenRouter` doesn't have any **traces**